# Vision Transformers (ViT)

_Modern Deep Learning & AI_

**Chop an image into patches, treat each patch as a word, and let attention mix them.**

Transformers conquered language by treating a sentence as a sequence of word tokens. A Vision Transformer does the same to images.

     It slices the image into a grid of small square patches. Each patch becomes one token, like a word.

---

This notebook is a practice scaffold for the **Vision Transformers (ViT)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn

class ViT(nn.Module):
    def __init__(self, img=32, patch=8, dim=64, depth=2, heads=4, n_cls=10):
        super().__init__()
        n_patches = (img // patch) ** 2                  # N = (H/P)*(W/P)
        # one strided conv = flatten-each-patch then linear-embed
        self.embed = nn.Conv2d(3, dim, kernel_size=patch, stride=patch)
        self.cls = nn.Parameter(torch.zeros(1, 1, dim))  # [CLS] token
        self.pos = nn.Parameter(torch.zeros(1, n_patches + 1, dim))
        enc = nn.TransformerEncoderLayer(dim, heads, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc, num_layers=depth)
        self.head = nn.Linear(dim, n_cls)

    def forward(self, x):                                 # x: (B, 3, H, W)
        t = self.embed(x).flatten(2).transpose(1, 2)     # (B, N, dim) tokens
        cls = self.cls.expand(x.size(0), -1, -1)
        t = torch.cat([cls, t], dim=1) + self.pos        # prepend CLS, add pos
        t = self.encoder(t)                              # attention over patches
        return self.head(t[:, 0])                        # classify from CLS

model = ViT()
x = torch.randn(2, 3, 32, 32)        # 2 synthetic RGB images
logits = model(x)                    # (2, 10) class scores
print(logits.shape)

## Visualize the data & results

_On a real handwritten 3 from load_digits(), which image patches does the CLS token attend to most?_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

digits = load_digits()
img = digits.data[digits.target == 3][0].reshape(8, 8)   # a real handwritten 3
imgp = np.pad(img, ((0, 1), (0, 1)), mode='edge')        # 9x9 so it tiles 3x3

# 9 patches of 3x3 pixels each.
patches = np.array([imgp[r*3:r*3+3, c*3:c*3+3].flatten()
                    for r in range(3) for c in range(3)], dtype=float)
pn = patches / (np.linalg.norm(patches, axis=1, keepdims=True) + 1e-9)
q = pn.mean(0); q = q / (np.linalg.norm(q) + 1e-9)        # CLS query

scores = pn @ q * 4.0                                     # scaled dot-product
w = np.exp(scores - scores.max()); w = w / w.sum()        # softmax -> sums to 1
grid = w.reshape(3, 3)

fig, ax = plt.subplots(1, 2, figsize=(9, 4))
im = ax[0].imshow(grid, cmap='magma')
ax[0].set_title('CLS attention over 3x3 patches of a real 3')
ax[0].set_xticks(range(3)); ax[0].set_yticks(range(3))
for i in range(3):
    for j in range(3):
        ax[0].text(j, i, format(grid[i, j], '.3f'), ha='center', va='center', color='w')
fig.colorbar(im, ax=ax[0])

cols = ['#ffb454' if i == int(np.argmax(w)) else '#4ea1ff' for i in range(9)]
ax[1].bar([f'p{i}' for i in range(9)], w, color=cols)
ax[1].set_title('Attention weight per patch token'); ax[1].set_ylabel('weight')
plt.tight_layout(); plt.show()